# Ising entanglement scan, L=16..24, ell=2..8

This notebook plots the direct puMPS/tangent-state scan saved in `ising_entanglement_scan_L16_24_variableD_ell2_8_direct.csv`. It follows the plotting style of `Ising_entanglement_scan_variableD.ipynb` and writes new PDF/PNG files without overwriting previous variable-D plots.

In [ ]:

import csv
import math
import os
from pathlib import Path
import numpy as np

mpl_config = Path("/private/tmp/pumps_matplotlib_cache")
mpl_config.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_config))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "CMU Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "font.size": 20,
    "axes.labelsize": 24,
    "axes.titlesize": 26,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 18,
    "lines.linewidth": 6,
    "lines.markersize": 50,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "axes.linewidth": 1.5,
    "grid.linestyle": "--",
    "grid.linewidth": 2.5,
})

root = Path.cwd()
if not (root / "examples").exists():
    root = root.parent
examples_dir = root / "examples"
csv_path = examples_dir / "ising_entanglement_scan_L16_24_variableD_ell2_8_direct.csv"

rows = []
with csv_path.open() as f:
    reader = csv.DictReader(f)
    for r in reader:
        rows.append({
            "N": int(r["N"]),
            "D": int(r["D"]),
            "ell": int(r["ell"]),
            "x": float(r["x"]),
            "E0": float(r["E0"]),
            "E1": float(r["E1"]),
            "E2": float(r["E2"]),
            "gap_ratio": float(r["gap_ratio"]),
            "P0": float(r["P0"]),
            "P1": float(r["P1"]),
            "P2": float(r["P2"]),
            "I01": float(r["I01"]),
            "I02": float(r["I02"]),
            "I12": float(r["I12"]),
            "S10": float(r["S10"]),
            "S20": float(r["S20"]),
            "S12": float(r["S12"]),
        })

print(f"loaded {len(rows)} rows from {csv_path}")
for N in sorted({r["N"] for r in rows}):
    row = next(r for r in rows if r["N"] == N)
    ells = [r["ell"] for r in rows if r["N"] == N]
    Dval = row["D"]
    gap_ratio = row["gap_ratio"]
    P0, P1, P2 = row["P0"], row["P1"], row["P2"]
    print(
        f"L={N:2d}, D={Dval:2d}, ell={min(ells)}:{max(ells)}, "
        f"gap_ratio={gap_ratio:.6f}, parity=({P0:.6f},{P1:.6f},{P2:.6f})"
    )

marker_cycle = ["o", "s", "^", "D", "v", "P", "X"]
pair_specs = [
    ("I01", r"$I$-$\sigma$", "tab:blue", marker_cycle[0]),
    ("I02", r"$I$-$\epsilon$", "tab:orange", marker_cycle[1]),
    ("I12", r"$\sigma$-$\epsilon$", "tab:green", marker_cycle[2]),
]
rel_specs = [
    ("S10", r"$S(\rho_\sigma\Vert\rho_I)$", "tab:blue", marker_cycle[0]),
    ("S20", r"$S(\rho_\epsilon\Vert\rho_I)$", "tab:orange", marker_cycle[1]),
    ("S12", r"$S(\rho_\sigma\Vert\rho_\epsilon)$", "tab:green", marker_cycle[2]),
]

def fit_power_law(xs, ys, fit_max_x=0.55):
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    mask = (xs > 0) & (ys > 0) & (xs < fit_max_x)
    xs = xs[mask]
    ys = ys[mask]
    if len(xs) < 2:
        raise ValueError("Need at least two positive points for the log-log fit.")
    beta = np.linalg.lstsq(np.column_stack([np.ones_like(xs), np.log(xs)]), np.log(ys), rcond=None)[0]
    return math.exp(beta[0]), beta[1], xs, ys

def plot_observable(specs, ylabel, filename, fit_max_x=0.55, fit_legend_label=None):
    fig, ax = plt.subplots(1, 1, figsize=(22, 16))
    fit_lines = []
    for key, label, color, marker in specs:
        xs = np.array([r["x"] for r in rows], dtype=float)
        ys = np.array([r[key] for r in rows], dtype=float)
        sort_idx = np.argsort(xs)
        xs = xs[sort_idx]
        ys = ys[sort_idx]
        ax.plot(xs, ys, linestyle="None", marker=marker, color=color, label=label, markersize=36)
        A, gamma, x_used, y_used = fit_power_law(xs, ys, fit_max_x=fit_max_x)
        x_line = np.logspace(np.log10(x_used.min() * 0.95), np.log10(x_used.max() * 1.05), 300)
        y_line = A * (x_line ** gamma)
        ax.plot(x_line, y_line, linestyle="--", alpha=0.8, color=color)
        fit_lines.append((label, A, gamma))
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"$x =Arc(Q_A)/2\pi$", fontsize=72)
    ax.set_ylabel(ylabel, fontsize=72)
    ax.tick_params(axis="both", labelsize=60)
    ax.set_xticks([0.1, 0.2, 0.3, 0.4, 0.5])
    ax.set_xticklabels([r"$0.1$", r"$0.2$", r"$0.3$", r"$0.4$", r"$0.5$"], fontsize=50)
    ax.grid(True, which="both")
    if fit_legend_label is not None:
        handles, labels = ax.get_legend_handles_labels()
        handles.append(Line2D([0], [0], color="black", linestyle="--", linewidth=6, alpha=0.8))
        labels.append(fit_legend_label)
        ax.legend(handles, labels, loc=(0.55, 0.25), fontsize=42)
    else:
        ax.legend(loc=(0.55, 0.25), fontsize=42)
    fit_text = "\n".join([rf"{label}: $min(\alpha,\beta)={gamma:.4f}$, $A={A:.4f}$" for label, A, gamma in fit_lines])
    ax.text(0.4, 0.08, fit_text, transform=ax.transAxes, fontsize=48,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.8))
    fig.tight_layout()
    pdf_path = examples_dir / f"{filename}.pdf"
    png_path = examples_dir / f"{filename}.png"
    fig.savefig(pdf_path, dpi=1200, bbox_inches="tight")
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("saved figure pdf:", pdf_path)
    print("saved figure png:", png_path)
    return {label: {"A": A, "gamma": gamma} for label, A, gamma in fit_lines}

mi_fits = plot_observable(
    pair_specs,
    r"$\Delta(Q_A;\mathbb{V}^{\text{CFT}})$",
    "ising_mutual_information_L16_24_variableD_ell2_8_direct",
    fit_legend_label=r"fit: $\Delta=A x^{min(\alpha,\beta)}$",
)
rel_fits = plot_observable(
    rel_specs,
    r"$S(\rho_A\Vert\rho_B)$",
    "ising_relative_entropy_L16_24_variableD_ell2_8_direct",
)
mi_fits, rel_fits
